# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Accessing dataset metadata attributes
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Authors: {dataset.metadata.author}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields.

In [ ]:
# List all record sets by `@id` and name
print("Record sets in the dataset:\n")
for record_set in dataset.metadata.record_sets:
    print(f"@id: {record_set.id}, name: {record_set.name}")

# For demonstration, list fields in the first record set
if dataset.metadata.record_sets:
    first_record_set = dataset.metadata.record_sets[0]
    print(f"\nFields for record set '{first_record_set.name}':")
    for field in first_record_set.fields:
        print(f"  @id: {field.id}, name: {field.name}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above. All references are to `@id` values.

In [ ]:
# Collect all record set `@id`s
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load each record set to a DataFrame
for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame with shape {df.shape}.")
    else:
        print(f"No records found for {record_set_id}.")

# Show columns for the first loaded DataFrame
main_record_set_id = next(iter(dataframes))  # pick the first one
print(f"\nColumns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps on the loaded DataFrame. Select numeric fields by their field `@id`, filter and normalize, and group by another field if available.

In [ ]:
# Define IDs for the main record set and numeric/group field
# For demonstration, we inspect the first loaded record set
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Find a numeric field (first field with dtype float or int)
numeric_field_id = None
for field in dataset.metadata.record_sets[0].fields:
    if field.data_type in ['Number', 'Float', 'Integer'] and field.id in df.columns:
        numeric_field_id = field.id
        break

if numeric_field_id is not None:
    print(f"Using numeric field '@id': {numeric_field_id}")
    # Filter records with the selected numeric field > threshold
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}\n")
    print(filtered_df.head())

    # Normalize the numeric field
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized values for '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Try grouping by a categorical field
    group_field_id = None
    for field in dataset.metadata.record_sets[0].fields:
        if field.data_type == 'Text' and field.id in df.columns:
            group_field_id = field.id
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by a text field if possible
    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for plotting.")

## 6. Conclusion
In this notebook, you have loaded and explored a Croissant-described dataset using `mlcroissant`, accessed record sets and fields by their `@id`s, performed basic EDA, and visualized numeric fields. For domain-specific insights, further analysis and statistical hypothesis testing can be performed.